# Установка зависимостей и импорт библиотек

In [ ]:
!pip install rectools[lightfm,torch] lightning-fabric

In [ ]:
import os
import glob
import json

import pandas as pd
import numpy as np

from rectools import Columns
from rectools.dataset import Dataset
from rectools.models import SASRecModel, BERT4RecModel
from rectools.metrics import NDCG, HitRate
from rectools.visuals.visual_app import ItemToItemVisualApp

from lightning_fabric import seed_everything

from sklearn.model_selection import train_test_split

In [ ]:
# Enable deterministic behaviour with CUDA >= 10.2
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
RANDOM_STATE = 42
seed_everything(RANDOM_STATE, workers=True)

INFO:lightning_fabric.utilities.seed:Seed set to 42


42

# Загрузка тренировочного датасета

Отберем только те элементы выборки, для которых трек прослушан более чем на 70%

In [ ]:
train_data = pd.read_csv('../hw/hw2/train_data.csv')

# Use rectools column names
train_interactions = train_data[
    train_data["time"] > 0.7
].copy().rename(columns={
    "user": Columns.User,
    "track": Columns.Item,
    "timestamp": Columns.Datetime,
    "time": Columns.Weight
})

train_interactions.head(3)

,message,datetime,user_id,item_id,weight,latency,recommendation,experiments
0,next,2026-04-25 04:38:05.535,4859,9495,1.00,0.001630,5268.0,{'HSTU': 'T1'}
1,next,2026-04-25 04:38:05.543,4097,3606,1.00,0.002550,14923.0,{'HSTU': 'C'}
11,next,2026-04-25 04:38:05.615,3806,10067,0.92,0.005116,953.0,{'HSTU': 'C'}


In [ ]:
train_interactions.shape

(193182, 8)

Добавим признаки из датасета треков. Выбранные признаки: имя исполнителя, его жанры, жанры трека, настроение трека и год его выпуска

In [ ]:
tracks = pd.read_json("../botify/data/tracks.json", lines=True).drop_duplicates(subset=["track"]).rename(columns={"track": Columns.Item})
tracks.head(1)

items = tracks.loc[tracks[Columns.Item].isin(train_interactions[Columns.Item])].copy()

artist = items[[Columns.Item, "artist"]].copy()
artist.columns = ["id", "value"]
artist["feature"] = "artist"

genre = items[[Columns.Item, "genres"]].explode("genres").copy()
genre.columns = ["id", "value"]
genre["feature"] = "genre"

artist_genre = items[[Columns.Item, "artist_genres"]].explode("artist_genres").copy()
artist_genre.columns = ["id", "value"]
artist_genre["feature"] = "artist_genre"

mood = items[[Columns.Item, "mood"]].copy()
mood.columns = ["id", "value"]
mood["feature"] = "mood"

year = items[[Columns.Item, "year"]].copy()
year.columns = ["id", "value"]
year["feature"] = "year"

item_features = pd.concat((artist, genre, artist_genre, mood, year))
item_features.head(3)

,id,value,feature
0,0,ABBA,artist
1,1,ABBA,artist
2,2,ABBA,artist


In [ ]:
train_dataset = Dataset.construct(
    interactions_df=train_interactions,
    item_features_df=item_features,
    cat_item_features=["artist", "genre", "artist_genre", "mood", "year"],
)

# Обучение моделей

Обучение SasRec

In [ ]:
sasrec = SASRecModel(
    session_max_len=100,
    loss="softmax",
    n_factors=64,
    n_blocks=1,
    n_heads=4,
    dropout_rate=0.2,
    lr=0.001,
    batch_size=128,
    epochs=100,
    verbose=1,
    deterministic=True,
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [ ]:
%%time
sasrec.fit(train_dataset)

/usr/local/lib/python3.12/dist-packages/rectools/dataset/identifiers.py:60: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  unq_values = pd.unique(values)
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='item_net_block_types', input_value=('rectools.models.nn.item...net.CatFeaturesItemNet'), input_type=tuple])
  return self.__pydantic_serializer__.to_python(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name        | Type                     | Params | Mode 
--

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.


CPU times: user 18min 6s, sys: 1.86 s, total: 18min 8s
Wall time: 18min 35s


Обучение Bert4Rec

In [ ]:
bert4rec = BERT4RecModel(
    session_max_len=100,
    mask_prob=0.15,
    loss="softmax",
    n_factors=64,
    n_blocks=1,
    n_heads=4,
    dropout_rate=0.2,
    lr=0.001,
    batch_size=128,
    epochs=200,
    verbose=1,
    deterministic=True,
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [ ]:
bert4rec.fit(train_dataset)

/usr/local/lib/python3.12/dist-packages/rectools/dataset/identifiers.py:60: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  unq_values = pd.unique(values)
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='item_net_block_types', input_value=('rectools.models.nn.item...net.CatFeaturesItemNet'), input_type=tuple])
  return self.__pydantic_serializer__.to_python(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name        | Type                     | Params | Mode 
--

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=200` reached.


# Получение предсказание для моделей на датасете

In [ ]:
item_ids = list(train_dataset.item_id_map.external_ids)[:15000]

In [ ]:
sasrec_recs = sasrec.recommend_to_items(
    target_items=item_ids,
    dataset=train_dataset,
    k=10,
    filter_itself=True,
    items_to_recommend=None,
    on_unsupported_targets="ignore"
)

In [ ]:
bert4rec_recs = bert4rec.recommend_to_items(
    target_items=item_ids,
    dataset=train_dataset,
    k=10,
    filter_itself=True,
    items_to_recommend=None,
    on_unsupported_targets="ignore"
)

# Ансамбль моделей

Соберем из ответов модели новый топ-10, взвешивая их ответы. Для того, чтобы корректно можно было сравнивать скоры от двух различных моделей, переведем их в "вероятности" с помощью softmax.

In [ ]:
def target_to_recommended(recs_df):
    result = {}
    for _, row in recs_df.iterrows():
        target = row['target_item_id']
        item = row['item_id']
        score = row['score']

        if target not in result:
            result[target] = {}
        result[target][item] = score

    return result

In [ ]:
def softmax(scores):
    scores = np.array(scores)
    scores = scores - scores.max()
    return np.exp(scores) / np.exp(scores).sum()

In [ ]:
recs = []
item_ids = list(train_dataset.item_id_map.external_ids)[:15000]
sasrec_weight = 0.4

sasrec_dict = target_to_recommended(sasrec_recs)
bert_dict = target_to_recommended(bert4rec_recs)

for item_id in item_ids:
    sasrec_dict_recs = sasrec_dict.get(item_id, {})
    bert_dict_recs = bert_dict.get(item_id, {})

    if not sasrec_dict_recs and not bert_dict_recs:
        continue

    recommended_items = set()

    for el in sasrec_dict_recs.keys():
      recommended_items.add(el)

    for el in bert_dict_recs.keys():
      recommended_items.add(el)

    items_list = []
    sasrec_scores = []
    bert_scores = []

    for item in recommended_items:
        items_list.append(item)
        sasrec_scores.append(sasrec_dict_recs.get(item, -1e9))
        bert_scores.append(bert_dict_recs.get(item, -1e9))

    sasrec_probs = softmax(sasrec_scores)
    lightfm_probs = softmax(bert_scores)
    final_scores = sasrec_weight * sasrec_probs + (1 - sasrec_weight) * lightfm_probs

    top_indices = np.argsort(final_scores)[::-1][:10]

    for rank, idx in enumerate(top_indices):
        recs.append({
            'target_item_id': item_id,
            'item_id': int(items_list[idx]),
            'rank': rank,
            'score': final_scores[idx],
        })

final_recs = pd.DataFrame(recs)

In [ ]:
final_recs

,target_item_id,item_id,rank,score
0,9495,2081,0,0.100995
1,9495,9627,1,0.100963
2,9495,2107,2,0.100671
3,9495,9622,3,0.098516
4,9495,2121,4,0.060053
...,...,...,...,...
149945,5094,10727,5,0.059971
149946,5094,10862,6,0.059939
149947,5094,10554,7,0.059868
149948,5094,10834,8,0.059843


# Сохранение результатов обучения

In [ ]:
def i2i_to_json(i2i_df, file_name):
    with open(file_name, "w") as json_file:
        for target_item_id, group in i2i_df.groupby('target_item_id'):
            recommendations = group.sort_values('rank')['item_id'].tolist()
            json_file.write(json.dumps({'item_id': target_item_id, 'recommendations': recommendations}) + "\n")


i2i_to_json(final_recs, "hw_recommendations.json")